In [ ]:
# Impor Library
import re
import pandas as pd
import numpy as np
from scipy import stats
from google.colab import files

In [ ]:
# Import File
up = files.upload()
nama_file = list(up.keys())[0]
print("File terbaca:", nama_file)

Saving Blank Quiz (Responses).xlsx to Blank Quiz (Responses) (1).xlsx
File terbaca: Blank Quiz (Responses) (1).xlsx


In [ ]:
df = pd.read_excel(nama_file)
kolom_butir = [c for c in df.columns if re.search(r"\[.*\]\s*$", c, re.S)]
butir = df[kolom_butir]
print("Jumlah responden :", len(df))
print("Jumlah butir     :", butir.shape[1])
print("Data kosong      :", int(butir.isna().sum().sum()))

Jumlah responden : 38
Jumlah butir     : 29
Data kosong      : 0


In [ ]:
# Mengubah jawaban Likert menjadi angka
skala = {
    "Sangat Tidak Setuju (STS)": 1,
    "Tidak Setuju (TS)": 2,
    "Netral (N)": 3,
    "Setuju (S)": 4,
    "Sangat Setuju (SS)": 5,
}
num = butir.replace(skala).apply(pd.to_numeric, errors="coerce")
print("Sel yang gagal dikonversi:", int(num.isna().sum().sum()))  # harus 0


Sel yang gagal dikonversi: 0


/tmp/ipykernel_3514/900392855.py:9: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  num = butir.replace(skala).apply(pd.to_numeric, errors="coerce")


In [ ]:
# Pengelompokkan butir per dimensi
dimensi = {}
for kolom in butir.columns:
    nama = re.match(r"^(.*?) \[", kolom, re.S).group(1).strip().rstrip(".").strip()
    dimensi.setdefault(nama, []).append(kolom)
for nama, daftar in dimensi.items():
    print(f"{nama}: {len(daftar)} butir")

# %% Fungsi inti
def r_tabel(n, alpha=0.05):
    t = stats.t.ppf(1 - alpha / 2, n - 2)
    return t / np.sqrt((n - 2) + t**2)

def cronbach_alpha(x):
    k = x.shape[1]
    return k / (k - 1) * (1 - x.var(ddof=1).sum() / x.sum(axis=1).var(ddof=1))

def kategori(a):
    return ("Sangat baik" if a >= 0.90 else "Baik" if a >= 0.80
            else "Reliabel" if a >= 0.70 else "Tidak reliabel")

def uji(data, dim):
    """Mengembalikan n, r tabel, tabel validitas, tabel reliabilitas."""
    n = len(data)
    rt = r_tabel(n)
    validitas, reliabilitas = [], []
    for nama, daftar in dim.items():
        x = data[daftar]
        total = x.sum(axis=1)
        a = cronbach_alpha(x)
        reliabilitas.append({"Dimensi": nama, "Jumlah butir": len(daftar),
                             "Cronbach's alpha": round(a, 3), "Kategori": kategori(a)})
        for i, kolom in enumerate(daftar, 1):
            r, p = stats.pearsonr(x[kolom], total)
            rc, _ = stats.pearsonr(x[kolom], total - x[kolom])
            validitas.append({
                "Dimensi": nama, "Butir": f"B{i}",
                "r hitung": round(r, 3), "Corrected r": round(rc, 3),
                "p-value": round(p, 4),
                "Keterangan": "Valid" if r > rt else "Tidak valid",
                "Alpha jika dihapus": (round(cronbach_alpha(x.drop(columns=kolom)), 3)
                                       if len(daftar) > 2 else np.nan),
            })
    return n, rt, pd.DataFrame(validitas), pd.DataFrame(reliabilitas)


Penggunaan AI pada tugas atau projek kuliah: 7 butir
Aspek Psikologis penggunaan AI pada tugas atau projek kuliah: 7 butir
Kebiasaan penggunaan teknologi digital: 6 butir
Prokrastinasi akademik: 4 butir
Rsspon emosional: 5 butir


In [ ]:
# %% Langkah 6-7: uji validitas dan reliabilitas
n, rt, tabel_validitas, tabel_reliabilitas = uji(num, dimensi)
print(f"n = {n} | r tabel (α=0,05; df={n-2}) = {rt:.3f}")
display(tabel_validitas)
display(tabel_reliabilitas)


n = 38 | r tabel (α=0,05; df=36) = 0.320


,Dimensi,Butir,r hitung,Corrected r,p-value,Keterangan,Alpha jika dihapus
0,Penggunaan AI pada tugas atau projek kuliah,B1,0.795,0.705,0.0000,Valid,0.837
1,Penggunaan AI pada tugas atau projek kuliah,B2,0.744,0.641,0.0000,Valid,0.846
2,Penggunaan AI pada tugas atau projek kuliah,B3,0.797,0.713,0.0000,Valid,0.836
3,Penggunaan AI pada tugas atau projek kuliah,B4,0.464,0.330,0.0033,Valid,0.881
4,Penggunaan AI pada tugas atau projek kuliah,B5,0.772,0.662,0.0000,Valid,0.843
5,Penggunaan AI pada tugas atau projek kuliah,B6,0.838,0.756,0.0000,Valid,0.829
6,Penggunaan AI pada tugas atau projek kuliah,B7,0.757,0.644,0.0000,Valid,0.846
7,Aspek Psikologis penggunaan AI pada tugas atau...,B1,0.861,0.800,0.0000,Valid,0.879
8,Aspek Psikologis penggunaan AI pada tugas atau...,B2,0.793,0.715,0.0000,Valid,0.889
9,Aspek Psikologis penggunaan AI pada tugas atau...,B3,0.763,0.662,0.0000,Valid,0.896


,Dimensi,Jumlah butir,Cronbach's alpha,Kategori
0,Penggunaan AI pada tugas atau projek kuliah,7,0.865,Baik
1,Aspek Psikologis penggunaan AI pada tugas atau...,7,0.904,Sangat baik
2,Kebiasaan penggunaan teknologi digital,6,0.777,Reliabel
3,Prokrastinasi akademik,4,0.851,Baik
4,Rsspon emosional,5,0.873,Baik


***LATIHAN***

In [ ]:
# Latihan 1: butir tidak valid dan butir terlemah
print("Butir tidak valid:")
display(tabel_validitas[tabel_validitas["Keterangan"] == "Tidak valid"])
print("5 butir dengan corrected r terendah:")
display(tabel_validitas.nsmallest(5, "Corrected r"))

Butir tidak valid:


,Dimensi,Butir,r hitung,Corrected r,p-value,Keterangan,Alpha jika dihapus


5 butir dengan corrected r terendah:


,Dimensi,Butir,r hitung,Corrected r,p-value,Keterangan,Alpha jika dihapus
3,Penggunaan AI pada tugas atau projek kuliah,B4,0.464,0.330,0.0033,Valid,0.881
19,Kebiasaan penggunaan teknologi digital,B6,0.629,0.451,0.0000,Valid,0.761
17,Kebiasaan penggunaan teknologi digital,B4,0.626,0.477,0.0000,Valid,0.756
15,Kebiasaan penggunaan teknologi digital,B2,0.701,0.494,0.0000,Valid,0.758
18,Kebiasaan penggunaan teknologi digital,B5,0.668,0.532,0.0000,Valid,0.745


In [ ]:
# Latihan 2: r hitung vs corrected r >= 0,30
print("Valid (r hitung > r tabel):", (tabel_validitas["Keterangan"] == "Valid").sum())
print("Lolos (corrected r >= 0,30):", (tabel_validitas["Corrected r"] >= 0.30).sum())
tabel_validitas["Selisih"] = tabel_validitas["r hitung"] - tabel_validitas["Corrected r"]
display(tabel_validitas.groupby("Dimensi")[["r hitung", "Corrected r", "Selisih"]].mean().round(3))

Valid (r hitung > r tabel): 29
Lolos (corrected r >= 0,30): 29


,r hitung,Corrected r,Selisih
Dimensi,,,
Aspek Psikologis penggunaan AI pada tugas atau projek kuliah,0.794,0.715,0.079
Kebiasaan penggunaan teknologi digital,0.690,0.530,0.160
Penggunaan AI pada tugas atau projek kuliah,0.738,0.636,0.102
Prokrastinasi akademik,0.839,0.703,0.135
Rsspon emosional,0.814,0.703,0.112


In [ ]:
# %% Latihan 3: buang satu butir, hitung ulang, bandingkan alpha
dim_target, no_butir = "Penggunaan AI pada tugas atau projek kuliah", 4  # ubah sesuai kebutuhan
dim_baru = {k: list(v) for k, v in dimensi.items()}
dibuang = dim_baru[dim_target].pop(no_butir - 1)
print("Butir dibuang:", dibuang)
_, _, val2, rel2 = uji(num, dim_baru)
banding = tabel_reliabilitas.merge(rel2, on="Dimensi", suffixes=(" sebelum", " sesudah"))
display(banding[banding["Dimensi"] == dim_target])
display(val2[val2["Dimensi"] == dim_target])

Butir dibuang: Penggunaan AI pada tugas atau projek kuliah [Saya merasa kesulitan memulai mengerjakan tugas tanpa bantuan AI.]


,Dimensi,Jumlah butir sebelum,Cronbach's alpha sebelum,Kategori sebelum,Jumlah butir sesudah,Cronbach's alpha sesudah,Kategori sesudah
0,Penggunaan AI pada tugas atau projek kuliah,7,0.865,Baik,6,0.881,Baik


,Dimensi,Butir,r hitung,Corrected r,p-value,Keterangan,Alpha jika dihapus
0,Penggunaan AI pada tugas atau projek kuliah,B1,0.796,0.699,0.0,Valid,0.858
1,Penggunaan AI pada tugas atau projek kuliah,B2,0.753,0.644,0.0,Valid,0.867
2,Penggunaan AI pada tugas atau projek kuliah,B3,0.820,0.738,0.0,Valid,0.853
3,Penggunaan AI pada tugas atau projek kuliah,B4,0.738,0.606,0.0,Valid,0.875
4,Penggunaan AI pada tugas atau projek kuliah,B5,0.865,0.789,0.0,Valid,0.842
5,Penggunaan AI pada tugas atau projek kuliah,B6,0.785,0.673,0.0,Valid,0.863


In [ ]:
# %% Latihan 4: pengaruh ukuran sampel
baris = []
for m in [10, 15, 20, 30, len(num)]:
    sub = num if m == len(num) else num.sample(m, random_state=1)
    _, rt_m, v_m, r_m = uji(sub, dimensi)
    baris.append({"n": m, "r tabel": round(rt_m, 3),
                  "Butir valid": int((v_m["Keterangan"] == "Valid").sum()),
                  "Butir tidak valid": int((v_m["Keterangan"] != "Valid").sum()),
                  "Corrected r >= 0,30": int((v_m["Corrected r"] >= 0.30).sum()),
                  "Alpha min": r_m["Cronbach's alpha"].min(),
                  "Alpha maks": r_m["Cronbach's alpha"].max()})
display(pd.DataFrame(baris))

,n,r tabel,Butir valid,Butir tidak valid,"Corrected r >= 0,30",Alpha min,Alpha maks
0,10,0.632,27,2,29,0.875,0.957
1,15,0.514,28,1,29,0.839,0.951
2,20,0.444,29,0,29,0.796,0.938
3,30,0.361,29,0,29,0.786,0.910
4,38,0.320,29,0,29,0.777,0.904


In [53]:
# Unduh Hasil
tabel_validitas = tabel_validitas.drop(columns="Selisih")
with pd.ExcelWriter("hasil_uji_validitas_reliabilitas.xlsx") as w:
    tabel_validitas.to_excel(w, sheet_name="Validitas", index=False)
    tabel_reliabilitas.to_excel(w, sheet_name="Reliabilitas", index=False)
files.download("hasil_uji_validitas_reliabilitas.xlsx")

KeyError: "['Selisih'] not found in axis"

In [54]:
!git config --global user.name "alonz-creator"
!git config --global user.email "salman24ti@mahasiswa.pcr.ac.id"

In [55]:
# Ganti USERNAME, REPO_NAME, dan TOKEN dengan Personal Access Token (PAT) Anda
!git clone https://<PAT_TOKEN>git@github.com:alonz-creator/Salman3TIE_Pengujian_Validitas_Reabilitas.git
%cd <Salman3TIE_Pengujian_Validitas_Reabilitas>

/bin/bash: line 1: PAT_TOKEN: No such file or directory
[Errno 2] No such file or directory: '<Salman3TIE_Pengujian_Validitas_Reabilitas>'
/content


In [57]:
!git add .
!git commit -m "Commit dari Google Colab"
!git push origin main

fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
